In [ ]:
import math
import time
from copy import deepcopy

class TicTacToeNxN:
    def __init__(self, N=5, use_alpha_beta=True):
        self.N = N
        self.board = [[' ' for _ in range(N)] for _ in range(N)]
        self.player = 'X'
        self.ai = 'O'
        self.use_alpha_beta = use_alpha_beta
        self.nodes_visited = 0
        self.nodes_pruned = 0

        # Ngưỡng chiều sâu để cắt tỉa (heuristic)
        self.max_depth = 4 if N > 5 else 6

    def print_board(self):
        print("\n  " + "   ".join(str(i % 10) for i in range(self.N)))
        for i in range(self.N):
            row_display = f"{i%10} "
            for j in range(self.N):
                row_display += self.board[i][j]
                if j < self.N - 1:
                    row_display += " | "
            print(row_display)
            if i < self.N - 1:
                print("  " + "-" * (self.N * 4 - 3))

    def is_winner(self, player):
        """Kiểm tra thắng trên bàn cờ NxN (cần 5 quân liên tiếp để thắng)"""
        win_count = 5  # Cần 5 quân liên tiếp để thắng

        # Kiểm tra hàng
        for i in range(self.N):
            count = 0
            for j in range(self.N):
                if self.board[i][j] == player:
                    count += 1
                    if count >= win_count:
                        return True
                else:
                    count = 0

        # Kiểm tra cột
        for j in range(self.N):
            count = 0
            for i in range(self.N):
                if self.board[i][j] == player:
                    count += 1
                    if count >= win_count:
                        return True
                else:
                    count = 0

        # Kiểm tra đường chéo chính
        for i in range(self.N - win_count + 1):
            for j in range(self.N - win_count + 1):
                if all(self.board[i + k][j + k] == player for k in range(win_count)):
                    return True

        # Kiểm tra đường chéo phụ
        for i in range(self.N - win_count + 1):
            for j in range(win_count - 1, self.N):
                if all(self.board[i + k][j - k] == player for k in range(win_count)):
                    return True

        return False

    def is_full(self):
        return all(self.board[i][j] != ' ' for i in range(self.N) for j in range(self.N))

    def is_game_over(self):
        return self.is_winner(self.player) or self.is_winner(self.ai) or self.is_full()

    def evaluate_heuristic(self):
        """
        Heuristic đánh giá bàn cờ cho NxN
        Dựa trên số lượng chuỗi 3,4 quân liên tiếp
        """
        if self.is_winner(self.ai):
            return 10000
        if self.is_winner(self.player):
            return -10000

        ai_score = 0
        player_score = 0
        win_count = 5

        # Hàm đánh giá một chuỗi
        def evaluate_sequence(seq):
            ai_count = seq.count(self.ai)
            player_count = seq.count(self.player)
            empty_count = seq.count(' ')

            if ai_count > 0 and player_count > 0:
                return 0

            if ai_count >= win_count:
                return 10000
            if player_count >= win_count:
                return -10000

            # Điểm cho AI
            if ai_count == 4 and empty_count > 0:
                return 1000
            if ai_count == 3 and empty_count > 0:
                return 100
            if ai_count == 2 and empty_count > 0:
                return 10

            # Điểm cho Player
            if player_count == 4 and empty_count > 0:
                return -1000
            if player_count == 3 and empty_count > 0:
                return -100
            if player_count == 2 and empty_count > 0:
                return -10

            return 0

        # Duyệt tất cả các hàng
        for i in range(self.N):
            for j in range(self.N - win_count + 1):
                seq = [self.board[i][j + k] for k in range(win_count)]
                score = evaluate_sequence(seq)
                if score > 0:
                    ai_score += score
                else:
                    player_score += abs(score)

        # Duyệt tất cả các cột
        for j in range(self.N):
            for i in range(self.N - win_count + 1):
                seq = [self.board[i + k][j] for k in range(win_count)]
                score = evaluate_sequence(seq)
                if score > 0:
                    ai_score += score
                else:
                    player_score += abs(score)

        # Duyệt đường chéo chính
        for i in range(self.N - win_count + 1):
            for j in range(self.N - win_count + 1):
                seq = [self.board[i + k][j + k] for k in range(win_count)]
                score = evaluate_sequence(seq)
                if score > 0:
                    ai_score += score
                else:
                    player_score += abs(score)

        # Duyệt đường chéo phụ
        for i in range(self.N - win_count + 1):
            for j in range(win_count - 1, self.N):
                seq = [self.board[i + k][j - k] for k in range(win_count)]
                score = evaluate_sequence(seq)
                if score > 0:
                    ai_score += score
                else:
                    player_score += abs(score)

        return ai_score - player_score

    def minimax(self, depth, is_maximizing, alpha=-math.inf, beta=math.inf):
        self.nodes_visited += 1

        if depth >= self.max_depth or self.is_game_over():
            return self.evaluate_heuristic()

        if is_maximizing:
            max_eval = -math.inf
            for i in range(self.N):
                for j in range(self.N):
                    if self.board[i][j] == ' ':
                        self.board[i][j] = self.ai
                        eval = self.minimax(depth + 1, False, alpha, beta)
                        self.board[i][j] = ' '
                        max_eval = max(max_eval, eval)

                        if self.use_alpha_beta:
                            alpha = max(alpha, eval)
                            if beta <= alpha:
                                self.nodes_pruned += 1
                                return max_eval
            return max_eval
        else:
            min_eval = math.inf
            for i in range(self.N):
                for j in range(self.N):
                    if self.board[i][j] == ' ':
                        self.board[i][j] = self.player
                        eval = self.minimax(depth + 1, True, alpha, beta)
                        self.board[i][j] = ' '
                        min_eval = min(min_eval, eval)

                        if self.use_alpha_beta:
                            beta = min(beta, eval)
                            if beta <= alpha:
                                self.nodes_pruned += 1
                                return min_eval
            return min_eval

    def best_move(self):
        best_score = -math.inf
        best_move = None
        alpha = -math.inf
        beta = math.inf
        self.nodes_visited = 0
        self.nodes_pruned = 0

        for i in range(self.N):
            for j in range(self.N):
                if self.board[i][j] == ' ':
                    self.board[i][j] = self.ai
                    score = self.minimax(0, False, alpha, beta)
                    self.board[i][j] = ' '
                    if score > best_score:
                        best_score = score
                        best_move = (i, j)
                    if self.use_alpha_beta:
                        alpha = max(alpha, score)

        return best_move

    def play(self):
        print(f"\n=== TIC TAC TOE {self.N}x{self.N} ===")
        print(f"Thuật toán: {'Alpha-Beta Pruning' if self.use_alpha_beta else 'Minimax'}")
        print(f"Độ sâu tối đa: {self.max_depth}")
        print("Bạn là X, AI là O")

        start_time = time.time()

        while not self.is_game_over():
            self.print_board()

            # Lượt người chơi
            while True:
                try:
                    row, col = map(int, input(f"\nLượt của bạn (0-{self.N-1}): ").split())
                    if 0 <= row < self.N and 0 <= col < self.N and self.board[row][col] == ' ':
                        self.board[row][col] = self.player
                        break
                    else:
                        print("Ô không hợp lệ!")
                except:
                    print(f"Nhập đúng định dạng: hàng cột (0-{self.N-1})")

            if self.is_game_over():
                break

            print("\nAI đang suy nghĩ...")
            move_start = time.time()
            move = self.best_move()
            move_time = time.time() - move_start

            if move:
                row, col = move
                self.board[row][col] = self.ai
                print(f"AI chọn ô ({row}, {col})")
                print(f"⏱ Thời gian suy nghĩ: {move_time:.2f}s")
                print(f"📊 Node đã xét: {self.nodes_visited}")
                if self.use_alpha_beta:
                    print(f"✂ Node bị cắt tỉa: {self.nodes_pruned}")

        total_time = time.time() - start_time
        self.print_board()

        if self.is_winner(self.ai):
            print(f"\n🔴 AI thắng sau {total_time:.2f}s!")
        elif self.is_winner(self.player):
            print(f"\n🟢 Bạn thắng sau {total_time:.2f}s! Xuất sắc!")
        else:
            print(f"\n⚪ Hòa sau {total_time:.2f}s!")

# Chạy thử nghiệm
if __name__ == "__main__":
    print("CHỌN KÍCH THƯỚC BÀN CỜ:")
    print("1. 5x5")
    print("2. 10x10")
    print("3. NxN (tùy chỉnh)")

    choice = input("Lựa chọn (1/2/3): ")

    if choice == '1':
        N = 5
    elif choice == '2':
        N = 10
    else:
        N = int(input("Nhập N (3-15): "))
        N = max(3, min(15, N))

    print("\nCHỌN THUẬT TOÁN:")
    print("1. Minimax thuần túy")
    print("2. Alpha-Beta Pruning")

    algo = input("Lựa chọn (1/2): ")
    use_ab = (algo == '2')

    game = TicTacToeNxN(N=N, use_alpha_beta=use_ab)
    game.play()

CHỌN KÍCH THƯỚC BÀN CỜ:
1. 5x5
2. 10x10
3. NxN (tùy chỉnh)
Lựa chọn (1/2/3): 1

CHỌN THUẬT TOÁN:
1. Minimax thuần túy
2. Alpha-Beta Pruning
Lựa chọn (1/2): 1

=== TIC TAC TOE 5x5 ===
Thuật toán: Minimax
Độ sâu tối đa: 6
Bạn là X, AI là O

  0   1   2   3   4
0   |   |   |   |  
  -----------------
1   |   |   |   |  
  -----------------
2   |   |   |   |  
  -----------------
3   |   |   |   |  
  -----------------
4   |   |   |   |  

Lượt của bạn (0-4): 0 1

AI đang suy nghĩ...


In [4]:
import math
import random
from abc import ABC, abstractmethod

class AlphaBetaGame(ABC):
    """Lớp trừu tượng cho các game sử dụng Alpha-Beta"""

    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.nodes_visited = 0
        self.nodes_pruned = 0

    @abstractmethod
    def get_initial_state(self):
        """Trả về trạng thái ban đầu của game"""
        pass

    @abstractmethod
    def get_legal_moves(self, state):
        """Trả về danh sách các nước đi hợp lệ"""
        pass

    @abstractmethod
    def make_move(self, state, move):
        """Thực hiện nước đi và trả về trạng thái mới"""
        pass

    @abstractmethod
    def is_terminal(self, state):
        """Kiểm tra xem game đã kết thúc chưa"""
        pass

    @abstractmethod
    def evaluate(self, state):
        """Đánh giá trạng thái (từ góc nhìn của người chơi MAX)"""
        pass

    @abstractmethod
    def print_state(self, state):
        """In trạng thái hiện tại"""
        pass

    def alpha_beta(self, state, depth, alpha, beta, is_maximizing):
        """Thuật toán Alpha-Beta Pruning tổng quát"""
        self.nodes_visited += 1

        if depth == 0 or self.is_terminal(state):
            return self.evaluate(state)

        if is_maximizing:
            max_eval = -math.inf
            for move in self.get_legal_moves(state):
                new_state = self.make_move(state, move)
                eval = self.alpha_beta(new_state, depth - 1, alpha, beta, False)
                max_eval = max(max_eval, eval)
                alpha = max(alpha, eval)
                if beta <= alpha:
                    self.nodes_pruned += 1
                    break
            return max_eval
        else:
            min_eval = math.inf
            for move in self.get_legal_moves(state):
                new_state = self.make_move(state, move)
                eval = self.alpha_beta(new_state, depth - 1, alpha, beta, True)
                min_eval = min(min_eval, eval)
                beta = min(beta, eval)
                if beta <= alpha:
                    self.nodes_pruned += 1
                    break
            return min_eval

    def best_move(self, state):
        """Tìm nước đi tốt nhất"""
        best_score = -math.inf
        best_move = None
        alpha = -math.inf
        beta = math.inf
        self.nodes_visited = 0
        self.nodes_pruned = 0

        for move in self.get_legal_moves(state):
            new_state = self.make_move(state, move)
            score = self.alpha_beta(new_state, self.max_depth - 1, alpha, beta, False)
            if score > best_score:
                best_score = score
                best_move = move
            alpha = max(alpha, score)

        return best_move, best_score


# ==================== CỜ VUA ĐƠN GIẢN ====================
class ChessGame(AlphaBetaGame):
    """Phiên bản cờ vua đơn giản hóa"""

    def __init__(self, max_depth=3):
        super().__init__(max_depth)
        self.board = None
        self.current_player = 'white'

    def get_initial_state(self):
        # Bàn cờ vua đơn giản (chỉ có vua và tốt)
        return {
            'board': [
                ['r', 'n', 'b', 'q', 'k', 'b', 'n', 'r'],
                ['p', 'p', 'p', 'p', 'p', 'p', 'p', 'p'],
                ['.', '.', '.', '.', '.', '.', '.', '.'],
                ['.', '.', '.', '.', '.', '.', '.', '.'],
                ['.', '.', '.', '.', '.', '.', '.', '.'],
                ['.', '.', '.', '.', '.', '.', '.', '.'],
                ['P', 'P', 'P', 'P', 'P', 'P', 'P', 'P'],
                ['R', 'N', 'B', 'Q', 'K', 'B', 'N', 'R']
            ],
            'turn': 'white'
        }

    def get_legal_moves(self, state):
        moves = []
        board = state['board']
        turn = state['turn']

        for i in range(8):
            for j in range(8):
                piece = board[i][j]
                if piece != '.':
                    if (turn == 'white' and piece.isupper()) or (turn == 'black' and piece.islower()):
                        moves.append(((i, j), (i+1, j)))  # Đơn giản hóa
        return moves[:10]  # Giới hạn số nước đi

    def make_move(self, state, move):
        new_state = {
            'board': [row[:] for row in state['board']],
            'turn': 'black' if state['turn'] == 'white' else 'white'
        }
        (from_i, from_j), (to_i, to_j) = move
        if 0 <= to_i < 8 and 0 <= to_j < 8:
            new_state['board'][to_i][to_j] = new_state['board'][from_i][from_j]
            new_state['board'][from_i][from_j] = '.'
        return new_state

    def is_terminal(self, state):
        # Kiểm tra chiếu hết đơn giản
        board = state['board']
        for i in range(8):
            for j in range(8):
                if board[i][j] == 'K' or board[i][j] == 'k':
                    return False
        return True

    def evaluate(self, state):
        """Đánh giá thế cờ"""
        piece_values = {
            'K': 1000, 'Q': 90, 'R': 50, 'B': 30, 'N': 30, 'P': 10,
            'k': -1000, 'q': -90, 'r': -50, 'b': -30, 'n': -30, 'p': -10
        }
        score = 0
        board = state['board']
        for i in range(8):
            for j in range(8):
                piece = board[i][j]
                if piece in piece_values:
                    score += piece_values[piece]
        return score if state['turn'] == 'white' else -score

    def print_state(self, state):
        print("\n   a b c d e f g h")
        for i in range(8):
            print(f"{8-i} ", end=" ")
            for j in range(8):
                print(state['board'][i][j], end=" ")
            print()


# ==================== CỜ TƯỚNG ĐƠN GIẢN ====================
class ChineseChessGame(AlphaBetaGame):
    """Phiên bản cờ tướng đơn giản hóa"""

    def get_initial_state(self):
        return {
            'board': [
                ['r', 'n', 'e', 'a', 'k', 'a', 'e', 'n', 'r'],
                ['.', '.', '.', '.', '.', '.', '.', '.', '.'],
                ['.', 'c', '.', '.', '.', '.', '.', 'c', '.'],
                ['p', '.', 'p', '.', 'p', '.', 'p', '.', 'p'],
                ['.', '.', '.', '.', '.', '.', '.', '.', '.'],
                ['.', '.', '.', '.', '.', '.', '.', '.', '.'],
                ['P', '.', 'P', '.', 'P', '.', 'P', '.', 'P'],
                ['.', 'C', '.', '.', '.', '.', '.', 'C', '.'],
                ['.', '.', '.', '.', '.', '.', '.', '.', '.'],
                ['R', 'N', 'E', 'A', 'K', 'A', 'E', 'N', 'R']
            ],
            'turn': 'red'
        }

    def get_legal_moves(self, state):
        moves = []
        board = state['board']
        turn = state['turn']

        for i in range(10):
            for j in range(9):
                piece = board[i][j]
                if piece != '.':
                    if (turn == 'red' and piece.isupper()) or (turn == 'black' and piece.islower()):
                        moves.append(((i, j), (i+1, j)))
        return moves[:10]

    def make_move(self, state, move):
        new_state = {
            'board': [row[:] for row in state['board']],
            'turn': 'black' if state['turn'] == 'red' else 'red'
        }
        (from_i, from_j), (to_i, to_j) = move
        if 0 <= to_i < 10 and 0 <= to_j < 9:
            new_state['board'][to_i][to_j] = new_state['board'][from_i][from_j]
            new_state['board'][from_i][from_j] = '.'
        return new_state

    def is_terminal(self, state):
        board = state['board']
        for i in range(10):
            for j in range(9):
                if board[i][j] == 'K' or board[i][j] == 'k':
                    return False
        return True

    def evaluate(self, state):
        piece_values = {
            'K': 1000, 'Q': 90, 'R': 50, 'C': 45, 'N': 30, 'E': 20, 'A': 20, 'P': 10,
            'k': -1000, 'q': -90, 'r': -50, 'c': -45, 'n': -30, 'e': -20, 'a': -20, 'p': -10
        }
        score = 0
        board = state['board']
        for i in range(10):
            for j in range(9):
                piece = board[i][j]
                if piece in piece_values:
                    score += piece_values[piece]
        return score if state['turn'] == 'red' else -score

    def print_state(self, state):
        print("\n   " + " ".join(str(i) for i in range(9)))
        for i in range(10):
            print(f"{i:2d} " + " ".join(state['board'][i]))


# ==================== CỜ VÂY ĐƠN GIẢN ====================
class GoGame(AlphaBetaGame):
    """Phiên bản cờ vây đơn giản trên bàn 5x5"""

    def __init__(self, max_depth=3):
        super().__init__(max_depth)
        self.size = 5

    def get_initial_state(self):
        return {
            'board': [['.' for _ in range(self.size)] for _ in range(self.size)],
            'turn': 'black'
        }

    def get_legal_moves(self, state):
        moves = []
        board = state['board']
        for i in range(self.size):
            for j in range(self.size):
                if board[i][j] == '.':
                    moves.append((i, j))
        return moves

    def make_move(self, state, move):
        new_state = {
            'board': [row[:] for row in state['board']],
            'turn': 'white' if state['turn'] == 'black' else 'black'
        }
        i, j = move
        new_state['board'][i][j] = 'B' if state['turn'] == 'black' else 'W'
        return new_state

    def is_terminal(self, state):
        # Kiểm tra bàn cờ đầy
        board = state['board']
        for i in range(self.size):
            for j in range(self.size):
                if board[i][j] == '.':
                    return False
        return True

    def evaluate(self, state):
        """Đánh giá dựa trên số quân và lãnh thổ"""
        black_count = 0
        white_count = 0
        board = state['board']

        for i in range(self.size):
            for j in range(self.size):
                if board[i][j] == 'B':
                    black_count += 1
                elif board[i][j] == 'W':
                    white_count += 1

        if state['turn'] == 'black':
            return black_count - white_count
        else:
            return white_count - black_count

    def print_state(self, state):
        print("\n   " + " ".join(str(i) for i in range(self.size)))
        for i in range(self.size):
            print(f"{i}  " + " ".join(state['board'][i]))


# ==================== CHẠY DEMO ====================
if __name__ == "__main__":
    print("=== ALPHA-BETA PRUNING CHO CÁC GAME CHIẾN THUẬT ===\n")
    print("Chọn game:")
    print("1. Cờ vua (đơn giản hóa)")
    print("2. Cờ tướng (đơn giản hóa)")
    print("3. Cờ vây 5x5 (đơn giản hóa)")

    choice = input("Lựa chọn (1/2/3): ")

    if choice == '1':
        game = ChessGame(max_depth=3)
        state = game.get_initial_state()
        print("\n=== CỜ VUA ===")
    elif choice == '2':
        game = ChineseChessGame(max_depth=3)
        state = game.get_initial_state()
        print("\n=== CỜ TƯỚNG ===")
    else:
        game = GoGame(max_depth=3)
        state = game.get_initial_state()
        print("\n=== CỜ VÂY ===")

    game.print_state(state)

    print(f"\nĐang tính nước đi tốt nhất với độ sâu {game.max_depth}...")
    move, score = game.best_move(state)

    print(f"\n✅ Nước đi tốt nhất: {move}")
    print(f"📊 Điểm đánh giá: {score}")
    print(f"📈 Số node đã xét: {game.nodes_visited}")
    print(f"✂️ Số node bị cắt tỉa: {game.nodes_pruned}")

=== ALPHA-BETA PRUNING CHO CÁC GAME CHIẾN THUẬT ===

Chọn game:
1. Cờ vua (đơn giản hóa)
2. Cờ tướng (đơn giản hóa)
3. Cờ vây 5x5 (đơn giản hóa)
Lựa chọn (1/2/3): 2

=== CỜ TƯỚNG ===

   0 1 2 3 4 5 6 7 8
 0 r n e a k a e n r
 1 . . . . . . . . .
 2 . c . . . . . c .
 3 p . p . p . p . p
 4 . . . . . . . . .
 5 . . . . . . . . .
 6 P . P . P . P . P
 7 . C . . . . . C .
 8 . . . . . . . . .
 9 R N E A K A E N R

Đang tính nước đi tốt nhất với độ sâu 3...

✅ Nước đi tốt nhất: ((7, 1), (8, 1))
📊 Điểm đánh giá: 30
📈 Số node đã xét: 210
✂️ Số node bị cắt tỉa: 26


In [ ]:
import pygame
import math

# Khởi tạo
pygame.init()

WIDTH, HEIGHT = 300, 300
LINE_WIDTH = 5
WIN = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Tic Tac Toe")

# Màu
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)

# Board
board = [[' ' for _ in range(3)] for _ in range(3)]
player = 'X'
ai = 'O'

# Vẽ bảng
def draw_board():
    WIN.fill(WHITE)
    pygame.draw.line(WIN, BLACK, (100, 0), (100, 300), LINE_WIDTH)
    pygame.draw.line(WIN, BLACK, (200, 0), (200, 300), LINE_WIDTH)
    pygame.draw.line(WIN, BLACK, (0, 100), (300, 100), LINE_WIDTH)
    pygame.draw.line(WIN, BLACK, (0, 200), (300, 200), LINE_WIDTH)

def draw_marks():
    font = pygame.font.SysFont(None, 80)
    for i in range(3):
        for j in range(3):
            if board[i][j] != ' ':
                text = font.render(board[i][j], True, BLACK)
                WIN.blit(text, (j*100 + 30, i*100 + 10))

def is_winner(p):
    for i in range(3):
        if all(board[i][j] == p for j in range(3)): return True
        if all(board[j][i] == p for j in range(3)): return True
    if all(board[i][i] == p for i in range(3)): return True
    if all(board[i][2-i] == p for i in range(3)): return True
    return False

def is_full():
    return all(board[i][j] != ' ' for i in range(3) for j in range(3))

def minimax(depth, is_max):
    if is_winner(ai): return 10 - depth
    if is_winner(player): return -10 + depth
    if is_full(): return 0

    if is_max:
        best = -math.inf
        for i in range(3):
            for j in range(3):
                if board[i][j] == ' ':
                    board[i][j] = ai
                    score = minimax(depth+1, False)
                    board[i][j] = ' '
                    best = max(best, score)
        return best
    else:
        best = math.inf
        for i in range(3):
            for j in range(3):
                if board[i][j] == ' ':
                    board[i][j] = player
                    score = minimax(depth+1, True)
                    board[i][j] = ' '
                    best = min(best, score)
        return best

def best_move():
    best_score = -math.inf
    move = None
    for i in range(3):
        for j in range(3):
            if board[i][j] == ' ':
                board[i][j] = ai
                score = minimax(0, False)
                board[i][j] = ' '
                if score > best_score:
                    best_score = score
                    move = (i, j)
    return move

# Game loop
running = True
draw_board()

while running:
    draw_marks()
    pygame.display.update()

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        if event.type == pygame.MOUSEBUTTONDOWN:
            x, y = pygame.mouse.get_pos()
            row = y // 100
            col = x // 100

            if board[row][col] == ' ':
                board[row][col] = player

                if not is_winner(player) and not is_full():
                    move = best_move()
                    if move:
                        board[move[0]][move[1]] = ai

pygame.quit()